In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CustomerDataProcessing").getOrCreate()

In [0]:
df = spark.read.format("csv").option("header","true").load("/Volumes/olist_spark_workspace/default/mysamplevolume/customers.csv")
## 2 task on  UI as 2 partitions hain, so data my csutomer.csv was split into 2 partitions - for 1 Task for each Partition

In [0]:
df.rdd.getNumPartitions()  # Not supported in shared clusters, consider alternative approaches like df.repartition(n).count() or df.mapInPandas()

---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-8293540456053168>, line 1
----> 1 df.rdd.getNumPartitions()

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:2363, in DataFrame.rdd(self)
   2361 @property
   2362 def rdd(self) -> "RDD[Row]":
-> 2363     raise PySparkNotImplementedError(
   2364         errorClass="NOT_IMPLEMENTED",
   2365         messageParameters={"feature": "rdd"},
   2366     )

PySparkNotImplementedError: [NOT_IMPLEMENTED] Using custom code using PySpark RDDs is not allowed on shared clusters. We suggest using mapInPandas or mapInArrow for the most common use cases. For more details on compatibility and limitations, check: https://learn.microsoft.com/azure/databricks/compute/access-mode-limitations#shared-access-mode-limitations-on-unity-catalog

In [0]:
df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    False|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     True|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     True|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    False|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows


In [0]:
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- is_active: string (nullable = true)



In [0]:
#convert datatypes
from pyspark.sql.functions import *
df = df.withColumn("registration_date", to_date(col("registration_date"), "yyyy-MM-dd"))\
    .withColumn('is_active',col("is_active").cast("boolean"))


In [0]:
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [0]:
df = df.fillna({'city':'Unknown','state':'Unknown','country':'Unknown'})

In [0]:
df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    false|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    false|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows


In [0]:
#now can extract year from the date easily-as converted to date type

df = df.withColumn('registration_year',year(col('registration_date')))\
    .withColumn('registration_month',month(col('registration_date')))


In [0]:
unique_cities = df.select(countDistinct('city')).collect()
unique_cities[0][0]

unique_states = df.select(countDistinct('state')).collect()
unique_states[0][0]

unique_countries = df.select(countDistinct('country')).collect()
unique_countries[0][0]

1

In [0]:
df.cache()

DataFrame[customer_id: string, name: string, city: string, state: string, country: string, registration_date: date, is_active: boolean, registration_year: int, registration_month: int]

In [0]:
df.unpersist()

DataFrame[customer_id: string, name: string, city: string, state: string, country: string, registration_date: date, is_active: boolean, registration_year: int, registration_month: int]

In [0]:
#each city how many customers
df.groupBy('city').count().orderBy(col('count').desc()).show()

+---------+-----+
|     city|count|
+---------+-----+
|     Pune| 2243|
|Hyderabad| 2242|
|  Kolkata| 2223|
|Bangalore| 2211|
|    Delhi| 2200|
|Ahmedabad| 2198|
|  Chennai| 2194|
|   Mumbai| 2142|
+---------+-----+



In [0]:
df.groupBy('state','country').count().orderBy(col('count').desc()).show()

+-----------+-------+-----+
|      state|country|count|
+-----------+-------+-----+
|      Delhi|  India| 2578|
|    Gujarat|  India| 2543|
| Tamil Nadu|  India| 2536|
|  Telangana|  India| 2520|
|West Bengal|  India| 2503|
|Maharashtra|  India| 2490|
|  Karnataka|  India| 2483|
+-----------+-------+-----+



In [0]:
#Pivot Table - Count of Active and Inactive Users per state

df.groupBy('state').pivot('is_active').count().show()

##vai jo sql interview questions karte vak saw that - pivot s used to convert rows to COL , 
#unpivot from  =col to rows

+-----------+-----+----+
|      state|false|true|
+-----------+-----+----+
|  Karnataka| 1207|1276|
| Tamil Nadu| 1284|1252|
|    Gujarat| 1211|1332|
|      Delhi| 1356|1222|
|  Telangana| 1294|1226|
|Maharashtra| 1260|1230|
|West Bengal| 1306|1197|
+-----------+-----+----+



In [0]:
##using case when
df.groupby('state').agg(
    sum(when(col('is_active') == "true",1).otherwise(0)).alias('active_customer'),
    sum(when(col('is_active')=="false",1).otherwise(0)).alias('inactive_customer')
).show()

+-----------+---------------+-----------------+
|      state|active_customer|inactive_customer|
+-----------+---------------+-----------------+
|  Karnataka|           1276|             1207|
| Tamil Nadu|           1252|             1284|
|    Gujarat|           1332|             1211|
|      Delhi|           1222|             1356|
|Maharashtra|           1230|             1260|
|West Bengal|           1197|             1306|
|  Telangana|           1226|             1294|
+-----------+---------------+-----------------+



In [0]:
#windowfunction
from pyspark.sql.window import Window

window_spec = Window.partitionBy('state').orderBy(col('registration_year').desc())

df = df.withColumn('rank',rank().over(window_spec))\
    .withColumn('dense_rank',dense_rank().over(window_spec))\
        .withColumn('row_number',row_number().over(window_spec))

In [0]:
df.show(5)

+-----------+-----------+-------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+
|customer_id|       name|   city|state|country|registration_date|is_active|registration_year|registration_month|rank|dense_rank|row_number|
+-----------+-----------+-------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+
|          6| Customer_6|   Pune|Delhi|  India|       2023-08-29|    false|             2023|                 8|   1|         1|         1|
|         18|Customer_18|   Pune|Delhi|  India|       2023-10-04|     true|             2023|                10|   1|         1|         2|
|         26|Customer_26|  Delhi|Delhi|  India|       2023-03-22|     true|             2023|                 3|   1|         1|         3|
|         46|Customer_46|Kolkata|Delhi|  India|       2023-09-23|     true|             2023|                 9|   1|         1|         4|
|         54|Custome

In [0]:
df.select('name','city','state','rank','dense_rank','row_number').show()

+------------+---------+-----+----+----------+----------+
|        name|     city|state|rank|dense_rank|row_number|
+------------+---------+-----+----+----------+----------+
|  Customer_6|     Pune|Delhi|   1|         1|         1|
| Customer_18|     Pune|Delhi|   1|         1|         2|
| Customer_26|    Delhi|Delhi|   1|         1|         3|
| Customer_46|  Kolkata|Delhi|   1|         1|         4|
| Customer_54|  Kolkata|Delhi|   1|         1|         5|
| Customer_61|Hyderabad|Delhi|   1|         1|         6|
| Customer_76|  Kolkata|Delhi|   1|         1|         7|
| Customer_82|  Kolkata|Delhi|   1|         1|         8|
| Customer_84|     Pune|Delhi|   1|         1|         9|
| Customer_92|  Kolkata|Delhi|   1|         1|        10|
|Customer_102|    Delhi|Delhi|   1|         1|        11|
|Customer_116|  Kolkata|Delhi|   1|         1|        12|
|Customer_119|Ahmedabad|Delhi|   1|         1|        13|
|Customer_129|  Kolkata|Delhi|   1|         1|        14|
|Customer_132|

In [0]:
#get recent customers

df_recent_customers = df.filter(col('registration_date') >= lit('2023-07-01'))
df_recent_customers.count()
df.count()

17653

In [0]:
df.count()

17653

In [0]:
## oldest and newest customer per city

df.groupBy('city').agg(
    min('registration_date').alias('oldest'),
    max('registration_date').alias('newest')
).show(5)

+---------+----------+----------+
|     city|    oldest|    newest|
+---------+----------+----------+
|Bangalore|2023-01-01|2023-12-31|
|  Chennai|2023-01-01|2023-12-31|
|   Mumbai|2023-01-01|2023-12-31|
|Ahmedabad|2023-01-01|2023-12-31|
|  Kolkata|2023-01-01|2023-12-31|
+---------+----------+----------+
only showing top 5 rows


In [0]:
output_path = '/Volumes/olist_spark_workspace/default/mysamplevolume/tables/processed_customers'
df.write.mode('overwrite').parquet(output_path)

## Joining and Analyzing customers and orders

In [0]:
orders_df = spark.read.format('csv').option('header','true').option('inferSchema','true').load('/Volumes/olist_spark_workspace/default/mysamplevolume/orders.csv')
orders_df

orders_df.show(5)
orders_df.printSchema()

+--------+-----------+----------+-----------------+---------+
|order_id|customer_id|order_date|     total_amount|   status|
+--------+-----------+----------+-----------------+---------+
|       0|       3692|2024-09-03|547.7160076008001|  Shipped|
|       1|      11055|2024-08-10|577.8942599188381|  Pending|
|       2|       6963|2024-08-22|484.2085562764487|  Pending|
|       3|      13268|2024-09-01|366.3286882431848|Cancelled|
|       4|       1131|2024-08-09|896.9588380686909|  Pending|
+--------+-----------+----------+-----------------+---------+
only showing top 5 rows
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- status: string (nullable = true)



In [0]:
customers_orders_df = df.join(orders_df,'customer_id',"inner")

In [0]:
customers_orders_df.count()

17653

In [0]:
customers_orders_df.show(5)

+-----------+-----------+---------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+--------+----------+------------------+---------+
|customer_id|       name|     city|state|country|registration_date|is_active|registration_year|registration_month|rank|dense_rank|row_number|order_id|order_date|      total_amount|   status|
+-----------+-----------+---------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+--------+----------+------------------+---------+
|          6| Customer_6|     Pune|Delhi|  India|       2023-08-29|    false|             2023|                 8|   1|         1|         1|   15577|2024-01-23| 890.4308117188951|  Pending|
|         18|Customer_18|     Pune|Delhi|  India|       2023-10-04|     true|             2023|                10|   1|         1|         2|   14717|2024-09-21|440.16719617331097|  Pending|
|         26|Customer_26|    Delhi|Delhi|  In

In [0]:
customers_orders_df.display()

customer_id,name,city,state,country,registration_date,is_active,registration_year,registration_month,rank,dense_rank,row_number,order_id,order_date,total_amount,status
6,Customer_6,Pune,Delhi,India,2023-08-29,false,2023,8,1,1,1,15577,2024-01-23,890.4308117188951,Pending
18,Customer_18,Pune,Delhi,India,2023-10-04,true,2023,10,1,1,2,14717,2024-09-21,440.16719617331097,Pending
26,Customer_26,Delhi,Delhi,India,2023-03-22,true,2023,3,1,1,3,4828,2024-02-12,747.061129639986,Pending
46,Customer_46,Kolkata,Delhi,India,2023-09-23,true,2023,9,1,1,4,17219,2024-05-30,999.592553819927,Cancelled
61,Customer_61,Hyderabad,Delhi,India,2023-12-31,false,2023,12,1,1,6,17283,2024-01-21,171.91694722937515,Pending
76,Customer_76,Kolkata,Delhi,India,2023-01-08,false,2023,1,1,1,7,5684,2024-09-15,684.7820270202418,Cancelled
102,Customer_102,Delhi,Delhi,India,2023-06-10,false,2023,6,1,1,11,16255,2024-07-10,402.0601254424326,Pending
116,Customer_116,Kolkata,Delhi,India,2023-06-17,true,2023,6,1,1,12,4681,2024-10-02,720.4991136269402,Delivered
129,Customer_129,Kolkata,Delhi,India,2023-07-20,true,2023,7,1,1,14,14508,2024-10-07,77.51947794718106,Shipped
151,Customer_151,Chennai,Delhi,India,2023-07-15,true,2023,7,1,1,16,5246,2024-04-05,268.0486561671143,Cancelled


In [0]:
#total orders per custoemr
customers_orders_count = customers_orders_df.groupBy('customer_id').count().orderBy(col('count').desc())
customers_orders_count.show(5)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|      11776|    7|
|       3243|    6|
|      14838|    6|
|       3884|    6|
|      13034|    6|
+-----------+-----+
only showing top 5 rows


In [0]:
#total spent per customer
customer_total_spend = customers_orders_df.groupBy('customer_id').agg(sum("total_amount")).orderBy(col('sum(total_amount)').desc())
customer_total_spend.show(5)


+-----------+------------------+
|customer_id| sum(total_amount)|
+-----------+------------------+
|       3336| 4362.550733141537|
|       3884|  4187.99763145619|
|      16020|3967.2692112582276|
|      14372| 3961.787139557334|
|      14933|3828.5841072418348|
+-----------+------------------+
only showing top 5 rows


In [0]:
#average spent per customer
customer_avg_spend = customers_orders_df.groupBy('customer_id').agg(sum("total_amount")).orderBy(col('sum(total_amount)').desc())
customer_avg_spend.show(5)

+-----------+------------------+
|customer_id| sum(total_amount)|
+-----------+------------------+
|       3336| 4362.550733141537|
|       3884|  4187.99763145619|
|      16020|3967.2692112582276|
|      14372| 3961.787139557334|
|      14933|3828.5841072418348|
+-----------+------------------+
only showing top 5 rows


In [0]:
#group by status

order_status_count = customers_orders_df.groupBy('status').count()
order_status_count.show(5)

+---------+-----+
|   status|count|
+---------+-----+
|  Shipped| 4386|
|Cancelled| 4469|
|Delivered| 4341|
|  Pending| 4457|
+---------+-----+



In [0]:
#order by month

order_by_month = customers_orders_df.withColumn('order_month',month(col("order_date")))\
    .groupBy('order_month')\
        .count()\
            .orderBy(col('order_month'))

order_by_month.show(5)

+-----------+-----+
|order_month|count|
+-----------+-----+
|          1| 1499|
|          2| 1368|
|          3| 1539|
|          4| 1457|
|          5| 1518|
+-----------+-----+
only showing top 5 rows


In [0]:
window_spec = Window.orderBy(col('sum(total_amount)').desc())

ranked_customers = customer_total_spend.withColumn('dense_rank',dense_rank().over(window_spec))
ranked_customers.show(10)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+------------------+----------+
|customer_id| sum(total_amount)|dense_rank|
+-----------+------------------+----------+
|       3336| 4362.550733141537|         1|
|       3884|  4187.99763145619|         2|
|      16020|3967.2692112582276|         3|
|      14372| 3961.787139557334|         4|
|      14933|3828.5841072418348|         5|
|       7566| 3647.119115720654|         6|
|      10559|3548.8378633460234|         7|
|      11776|  3438.36692751212|         8|
|      11449| 3396.060974816134|         9|
|       5425| 3389.162933156913|        10|
+-----------+------------------+----------+
only showing top 10 rows


In [0]:
customer_total_spend.printSchema() , customers_orders_count.printSchema()



root
 |-- customer_id: string (nullable = true)
 |-- sum(total_amount): double (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- count: long (nullable = false)



(None, None)

In [0]:
#finding customer with high order frequency but low total spend 

## so for tehse custoemrs we can give fomore discount and all
customer_spend_vs_orders = customers_orders_count.join(customer_total_spend, 'customer_id','inner' )\
    .orderBy(col('count').desc(),col('sum(total_amount)').asc())

customer_spend_vs_orders.show(5)



+-----------+-----+------------------+
|customer_id|count| sum(total_amount)|
+-----------+-----+------------------+
|      11776|    7|  3438.36692751212|
|       5160|    6| 1656.737343311546|
|       4294|    6| 1821.603928366352|
|       3243|    6|2860.1827303387754|
|      14838|    6| 2894.355602564058|
+-----------+-----+------------------+
only showing top 5 rows


In [0]:
output_path = '/Volumes/olist_spark_workspace/default/mysamplevolume/tables/final_customers_orders'
customers_orders_df.write.mode('overwrite').parquet(output_path)